In [1]:
import pyspark
from pyspark.sql import SparkSession

In [110]:
spark = SparkSession.builder\
        .master("local[*]")\
        .appName("test")\
        .getOrCreate()

In [54]:
df_green = spark.read.option("recursiveFileLookup","true").parquet("data/pq/green")

In [58]:
df_yellow = spark.read.option("recursiveFileLookup","true").parquet("data/pq/yellow")

### Renaming Columns

In [113]:
df_yellow = df_yellow\
            .withColumnRenamed('tpep_pickup_datetime','pickup_datetime')\
            .withColumnRenamed('tpep_dropoff_datetime','dropoff_datetime')

In [114]:
df_green = df_green\
            .withColumnRenamed('lpep_pickup_datetime','pickup_datetime')\
            .withColumnRenamed('lpep_dropoff_datetime','dropoff_datetime')

### Getting similar Columns From The Green And Yellow Dataset

In [82]:
common_columns = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_columns.append(col)

In [88]:
from pyspark.sql import functions as F

In [91]:
df_green_sel = df_green\
    .select(common_columns)\
    .withColumn('service_type',F.lit('green'))

In [132]:
df_yellow_sel = df_yellow\
    .select(common_columns)\
    .withColumn('service_type',F.lit('yellow'))

In [133]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [96]:
df_trips_data.groupBy('service_type').count().show()

[Stage 50:================================================>       (13 + 2) / 15]

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [98]:
df_trips_data.registerTempTable('trips_data')

In [102]:
spark.sql("""
SELECT 
    service_type,
    count(1)
FROM 
    trips_data
GROUP BY 
    service_type
""").show()

[Stage 53:====================================================>   (14 + 1) / 15]

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [106]:
df_select = spark.sql("""
SELECT 
    -- Reveneue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_montly_passenger_count,
    AVG(trip_distance) AS avg_montly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3

""")

In [109]:
df_select.coalesce(1).write.parquet('data/report/revenue',mode='overwrite')